# P6 MovieLens-1M recommender: Kaggle runner

Thin wrapper: all logic lives in `src/movierec`.

Settings: **GPU T4, Internet ON**, then *Save Version -> Save & Run All*. The run can be resumed: rerunning skips
(model, seed) rows that already finished.

Download `p6_results.zip` from the Output tab. It holds the metrics, segment and bootstrap tables, the FAISS
benchmark, the error analysis and `serve/` (seed-0 two-tower checkpoint + item vectors for the API). Per-user
arrays and the raw data are left out. MovieLens may not be redistributed: keep `serve/meta.json` out of git.

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/06-two-tower-recsys
!pip install -q -r requirements-gpu.txt && pip install -q -e . --no-deps

In [ ]:
!mkdir -p /kaggle/working/outputs/ml1m
!python -c "import torch, numpy, pandas, scipy, faiss, fastapi; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('numpy', numpy.__version__); print('pandas', pandas.__version__); print('scipy', scipy.__version__); print('faiss', faiss.__version__); print('fastapi', fastapi.__version__)" | tee /kaggle/working/outputs/ml1m/env.txt
!(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo no-gpu) | tee -a /kaggle/working/outputs/ml1m/env.txt
!lscpu | grep 'Model name' | tee -a /kaggle/working/outputs/ml1m/env.txt

## Run `configs/ml1m.yaml`
Downloads ml-1m.zip from GroupLens, splits by time, trains every model x seed, then writes the reports and the FAISS benchmark.

In [ ]:
!python scripts/run_experiment.py --config configs/ml1m.yaml --set output_dir=/kaggle/working/outputs/ml1m

In [ ]:
from pathlib import Path

for f in ['data_summary.md', 'results.md', 'segments.md', 'bootstrap.md', 'index.md', 'error_analysis.md']:
    p = Path('/kaggle/working/outputs/ml1m') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Package results
The zip holds text results and the serving artefacts (no per-user arrays).

In [ ]:
%cd /kaggle/working
!zip -qr p6_results.zip outputs -x '*/per_user/*'
!rm -rf outputs Artificial_Intelligence && ls -la /kaggle/working